In [1]:
!pip install mamba-ssm --no-build-isolation
!pip install causal-conv1d --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 2.9 MB/s eta 0:00:0000:0100:01
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 41.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 MB 20.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.4/772.4 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 58.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 89.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 21.8 MB/s eta 0:00:00
  Created wheel for mamba-ssm: 

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/models/ojasgupta074927/wikitext103-mamba-1-10500-32batch-size/pytorch/default/1/latest.pt


In [4]:
import os
import time
import math
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer

try:
    from mamba_ssm import Mamba as _CudaMamba
    HAS_CUDA_MAMBA = True
except ImportError:
    HAS_CUDA_MAMBA = False

class PureTorchSelectiveScan(nn.Module):
    """Unfused, pure-PyTorch selective SSM scan. Correctness-check only."""
    def __init__(self, d_inner: int, d_state: int = 16):
        super().__init__()
        self.d_inner, self.d_state = d_inner, d_state
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_inner, 1)))
        self.D = nn.Parameter(torch.ones(d_inner))
        self.x_proj = nn.Linear(d_inner, d_state * 2 + d_inner, bias=False)
        self.dt_proj = nn.Linear(d_inner, d_inner, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B_, L, D_ = x.shape
        A = -torch.exp(self.A_log.float())
        proj = self.x_proj(x)
        Bmat, Cmat, dt_in = torch.split(proj, [self.d_state, self.d_state, self.d_inner], dim=-1)
        delta = F.softplus(self.dt_proj(dt_in))
        h = torch.zeros(B_, D_, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dA = torch.exp(delta[:, t].unsqueeze(-1) * A)
            dBx = delta[:, t].unsqueeze(-1) * Bmat[:, t].unsqueeze(1) * x[:, t].unsqueeze(-1)
            h = dA * h + dBx
            ys.append((h * Cmat[:, t].unsqueeze(1)).sum(-1) + self.D * x[:, t])
        return torch.stack(ys, dim=1)

class MambaBlock(nn.Module):
    """Causal Mamba block, pre-norm + residual."""
    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2, use_grad_checkpoint: bool = True):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.use_grad_checkpoint = use_grad_checkpoint
        d_inner = expand * d_model
        if HAS_CUDA_MAMBA:
            self.mamba = _CudaMamba(d_model=d_model, d_state=d_state, expand=expand)
        else:
            self.in_proj = nn.Linear(d_model, d_inner)
            self.out_proj = nn.Linear(d_inner, d_model)
            self.scan = PureTorchSelectiveScan(d_inner, d_state)

    def _forward_impl(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        if HAS_CUDA_MAMBA:
            out = self.mamba(h)
        else:
            out = self.out_proj(self.scan(self.in_proj(h)))
        return x + out

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_grad_checkpoint and self.training:
            return torch.utils.checkpoint.checkpoint(self._forward_impl, x, use_reentrant=False)
        return self._forward_impl(x)

@dataclass
class MambaLMConfig:
    vocab_size: int = 50257
    block_size: int = 1024
    d_model: int = 384
    n_layer: int = 16
    d_state: int = 16
    expand: int = 2
    use_grad_checkpoint: bool = True

class MambaLM(nn.Module):
    def __init__(self, cfg: MambaLMConfig):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([
            MambaBlock(cfg.d_model, d_state=cfg.d_state, expand=cfg.expand,
                       use_grad_checkpoint=cfg.use_grad_checkpoint)
            for _ in range(cfg.n_layer)
        ])
        self.norm_out = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight

        nn.init.normal_(self.embed.weight, mean=0.0, std=0.02)

        if not HAS_CUDA_MAMBA:
            self.apply(self._init_weights)
            for name, p in self.named_parameters():
                if name.endswith("out_proj.weight"):
                    nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None):
        x = self.embed(input_ids)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm_out(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss

def get_memory_usage(device: str):
    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        process = psutil.Process(os.getpid())
        return process.memory_info().rss / (1024 ** 2)

@torch.no_grad()
def benchmark_inference_speed_and_memory(model, device, seq_lengths=[128, 256, 512, 1024], batch_size=1, warmup=3, runs=10):
    model.eval()
    model.to(device)
    
    print(f"\n--- Benchmarking Speed & Memory ({device.upper()}) ---")
    print(f"{'Seq Len':<10} | {'Latency (ms)':<15} | {'Throughput (tok/s)':<20} | {'Peak Memory (MB)':<15}")
    print("-" * 65)

    for seq_len in seq_lengths:
        input_ids = torch.randint(0, model.cfg.vocab_size, (batch_size, seq_len), device=device)
        
        if device == "cuda":
            torch.cuda.reset_peak_memory_stats()
        
        for _ in range(warmup):
            _ = model(input_ids)
            if device == "cuda": torch.cuda.synchronize()

        start_time = time.time()
        for _ in range(runs):
            _ = model(input_ids)
            if device == "cuda": torch.cuda.synchronize()
        end_time = time.time()

        avg_latency = (end_time - start_time) / runs
        throughput = (batch_size * seq_len) / avg_latency
        mem_usage = get_memory_usage(device)

        print(f"{seq_len:<10} | {avg_latency * 1000:<15.2f} | {throughput:<20.2f} | {mem_usage:<15.2f}")

@torch.no_grad()
def calculate_validation_perplexity(model, tokenizer, device, dataset_config="wikitext-103-raw-v1", stride=512):
    model.eval()
    model.to(device)
    
    print(f"\n--- Calculating Validation Perplexity ({device.upper()}) ---")
    val_data = load_dataset("wikitext", dataset_config, split="validation")
    encodings = tokenizer("\n\n".join(val_data["text"]), return_tensors="pt")
    
    seq_len = encodings.input_ids.size(1)
    max_length = model.cfg.block_size
    nlls = []
    prev_end_loc = 0
    
    for begin_loc in tqdm(range(0, seq_len, stride), desc="Evaluating"):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc 
        
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100
        
        outputs, _ = model(input_ids)
        shift_logits = outputs[0, :-1, :].contiguous()
        shift_labels = target_ids[0, 1:].contiguous()
        
        loss = F.cross_entropy(shift_logits, shift_labels, ignore_index=-100)
        nlls.append(loss)
        
        prev_end_loc = end_loc
        if end_loc == seq_len:
            break
            
    ppl = torch.exp(torch.stack(nlls).mean())
    print(f"Validation Perplexity: {ppl.item():.2f}")

@torch.no_grad()
def generate_samples(model, tokenizer, device, prompts, max_new_tokens=50, temperature=0.8):
    model.eval()
    model.to(device)
    print(f"\n--- Qualitative Generation ({device.upper()}) ---")
    
    for prompt in prompts:
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
        for _ in range(max_new_tokens):
            logits, _ = model(input_ids)
            next_token_logits = logits[:, -1, :] / temperature
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=-1)
            
        generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
        print(f"Prompt: {prompt}")
        print(f"Output: {generated_text}\n")

def main():
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    cfg = MambaLMConfig(
        vocab_size=tokenizer.vocab_size, 
        block_size=512,
        d_model=384, 
        n_layer=16,
        use_grad_checkpoint=False
    )
    
    model = MambaLM(cfg)
    ckpt_path = "/kaggle/input/models/ojasgupta074927/wikitext103-mamba-1-10500-32batch-size/pytorch/default/1/latest.pt"
    
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location="cpu")
        state_dict = ckpt["model_state"]
        if list(state_dict.keys())[0].startswith("module."):
            state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(state_dict)
        print(f"Loaded checkpoint from step {ckpt.get('step', 'unknown')}")
    else:
        print("Checkpoint not found. Running benchmarks on randomly initialized weights.")

    devices = ["cpu"]
    if torch.cuda.is_available():
        devices.append("cuda")

    test_prompts = [
        "The history of machine learning began with",
        "In a surprising turn of events, the global economy",
    ]

    for device in devices:
        # Prevent the RuntimeError caused by custom C++ CUDA kernels executing on CPU
        if device == "cpu" and HAS_CUDA_MAMBA:
            print(f"\n{'='*65}\n[NOTICE] Skipping CPU benchmarks.")
            print("The model is using the fused `mamba_ssm` backend. Its underlying ")
            print("C++ kernels (`causal_conv1d_cuda` and `selective_scan_fn`) are ")
            print("strictly written for GPU and will throw a RuntimeError on CPU.")
            print(f"{'='*65}")
            continue

        benchmark_inference_speed_and_memory(model, device)
        generate_samples(model, tokenizer, device, test_prompts)
        calculate_validation_perplexity(model, tokenizer, device, stride=256)

if __name__ == "__main__":
    main()

Loaded checkpoint from step 10500

[NOTICE] Skipping CPU benchmarks.
The model is using the fused `mamba_ssm` backend. Its underlying 
C++ kernels (`causal_conv1d_cuda` and `selective_scan_fn`) are 
strictly written for GPU and will throw a RuntimeError on CPU.

--- Benchmarking Speed & Memory (CUDA) ---
Seq Len    | Latency (ms)    | Throughput (tok/s)   | Peak Memory (MB)
-----------------------------------------------------------------
128        | 13.70           | 9345.79              | 167.48         
256        | 15.38           | 16645.17             | 193.13         
512        | 16.03           | 31937.31             | 241.67         
1024       | 22.27           | 45985.15             | 340.58         

--- Qualitative Generation (CUDA) ---
Prompt: The history of machine learning began with
Output: The history of machine learning began with the first of his two best books , The Washing Show regulars : The Liberal Party and The War of the Lions , in which the young American D

README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (251048 > 1024). Running this sequence through the model will result in indexing errors
Evaluating: 100%|█████████▉| 979/981 [00:16<00:00, 59.73it/s]


Validation Perplexity: 80.76


In [5]:
import os
import time
import math
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer

try:
    from mamba_ssm import Mamba as _CudaMamba
    HAS_CUDA_MAMBA = True
except ImportError:
    HAS_CUDA_MAMBA = False

class PureTorchSelectiveScan(nn.Module):
    """Unfused, pure-PyTorch selective SSM scan. Correctness-check only."""
    def __init__(self, d_inner: int, d_state: int = 16):
        super().__init__()
        self.d_inner, self.d_state = d_inner, d_state
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_inner, 1)))
        self.D = nn.Parameter(torch.ones(d_inner))
        self.x_proj = nn.Linear(d_inner, d_state * 2 + d_inner, bias=False)
        self.dt_proj = nn.Linear(d_inner, d_inner, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B_, L, D_ = x.shape
        A = -torch.exp(self.A_log.float())
        proj = self.x_proj(x)
        Bmat, Cmat, dt_in = torch.split(proj, [self.d_state, self.d_state, self.d_inner], dim=-1)
        delta = F.softplus(self.dt_proj(dt_in))
        h = torch.zeros(B_, D_, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dA = torch.exp(delta[:, t].unsqueeze(-1) * A)
            dBx = delta[:, t].unsqueeze(-1) * Bmat[:, t].unsqueeze(1) * x[:, t].unsqueeze(-1)
            h = dA * h + dBx
            ys.append((h * Cmat[:, t].unsqueeze(1)).sum(-1) + self.D * x[:, t])
        return torch.stack(ys, dim=1)


def mamba_pure_pytorch_forward(mamba_module, x):
    """
    Pure PyTorch equivalent of the mamba_ssm forward pass. 
    Uses the exact weights from the fused module but runs sequentially.
    """
    B, L, D = x.shape
    
    # 1. Input Projection
    xz = mamba_module.in_proj(x) 
    x_in, z = xz.chunk(2, dim=-1) 
    
    # 2. 1D Convolution
    x_in = x_in.transpose(1, 2)
    x_in = F.pad(x_in, (mamba_module.d_conv - 1, 0))
    x_in = mamba_module.conv1d(x_in)
    x_in = x_in.transpose(1, 2) 
    x_in = F.silu(x_in)
    
    # 3. State Projections (dt, B, C)
    x_proj = mamba_module.x_proj(x_in)
    dt, B_mat, C_mat = torch.split(
        x_proj, 
        [mamba_module.dt_rank, mamba_module.d_state, mamba_module.d_state], 
        dim=-1
    )
    
    # 4. dt Projection
    dt = F.softplus(mamba_module.dt_proj(dt)) 
    
    # 5. Selective Scan
    A = -torch.exp(mamba_module.A_log.float()) 
    D_mat = mamba_module.D.float() 
    
    h = torch.zeros(B, mamba_module.d_inner, mamba_module.d_state, device=x.device, dtype=x.dtype)
    ys = []
    
    for t in range(L):
        dt_t = dt[:, t].unsqueeze(-1)
        dA = torch.exp(dt_t * A)
        dBx = dt_t * B_mat[:, t].unsqueeze(1) * x_in[:, t].unsqueeze(-1)
        h = dA * h + dBx
        y_t = (h * C_mat[:, t].unsqueeze(1)).sum(dim=-1) + D_mat * x_in[:, t]
        ys.append(y_t)
        
    y = torch.stack(ys, dim=1)
    
    # 6. Output Projection
    out = y * F.silu(z)
    out = mamba_module.out_proj(out)
    return out


class MambaBlock(nn.Module):
    """Causal Mamba block, pre-norm + residual."""
    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2, use_grad_checkpoint: bool = True):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.use_grad_checkpoint = use_grad_checkpoint
        d_inner = expand * d_model
        if HAS_CUDA_MAMBA:
            self.mamba = _CudaMamba(d_model=d_model, d_state=d_state, expand=expand)
        else:
            self.in_proj = nn.Linear(d_model, d_inner)
            self.out_proj = nn.Linear(d_inner, d_model)
            self.scan = PureTorchSelectiveScan(d_inner, d_state)

    def _forward_impl(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        if HAS_CUDA_MAMBA:
            if x.is_cuda:
                out = self.mamba(h)
            else:
                out = mamba_pure_pytorch_forward(self.mamba, h)
        else:
            out = self.out_proj(self.scan(self.in_proj(h)))
        return x + out

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_grad_checkpoint and self.training:
            return torch.utils.checkpoint.checkpoint(self._forward_impl, x, use_reentrant=False)
        return self._forward_impl(x)

@dataclass
class MambaLMConfig:
    vocab_size: int = 50257
    block_size: int = 1024
    d_model: int = 384
    n_layer: int = 16
    d_state: int = 16
    expand: int = 2
    use_grad_checkpoint: bool = True

class MambaLM(nn.Module):
    def __init__(self, cfg: MambaLMConfig):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([
            MambaBlock(cfg.d_model, d_state=cfg.d_state, expand=cfg.expand,
                       use_grad_checkpoint=cfg.use_grad_checkpoint)
            for _ in range(cfg.n_layer)
        ])
        self.norm_out = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight

        nn.init.normal_(self.embed.weight, mean=0.0, std=0.02)

        if not HAS_CUDA_MAMBA:
            self.apply(self._init_weights)
            for name, p in self.named_parameters():
                if name.endswith("out_proj.weight"):
                    nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None):
        x = self.embed(input_ids)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm_out(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss

def get_memory_usage(device: str):
    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        process = psutil.Process(os.getpid())
        return process.memory_info().rss / (1024 ** 2)

@torch.no_grad()
def benchmark_inference_speed_and_memory(model, device, seq_lengths=[128, 256, 512, 1024], batch_size=1, warmup=3, runs=10):
    model.eval()
    model.to(device)
    
    print(f"\n--- Benchmarking Speed & Memory ({device.upper()}) ---")
    print(f"{'Seq Len':<10} | {'Latency (ms)':<15} | {'Throughput (tok/s)':<20} | {'Peak Memory (MB)':<15}")
    print("-" * 65)

    for seq_len in seq_lengths:
        input_ids = torch.randint(0, model.cfg.vocab_size, (batch_size, seq_len), device=device)
        
        if device == "cuda":
            torch.cuda.reset_peak_memory_stats()
        
        for _ in range(warmup):
            _ = model(input_ids)
            if device == "cuda": torch.cuda.synchronize()

        start_time = time.time()
        for _ in range(runs):
            _ = model(input_ids)
            if device == "cuda": torch.cuda.synchronize()
        end_time = time.time()

        avg_latency = (end_time - start_time) / runs
        throughput = (batch_size * seq_len) / avg_latency
        mem_usage = get_memory_usage(device)

        print(f"{seq_len:<10} | {avg_latency * 1000:<15.2f} | {throughput:<20.2f} | {mem_usage:<15.2f}")

@torch.no_grad()
def calculate_validation_perplexity(model, tokenizer, device, dataset_config="wikitext-103-raw-v1", stride=512):
    model.eval()
    model.to(device)
    
    print(f"\n--- Calculating Validation Perplexity ({device.upper()}) ---")
    val_data = load_dataset("wikitext", dataset_config, split="validation")
    encodings = tokenizer("\n\n".join(val_data["text"]), return_tensors="pt")
    
    seq_len = encodings.input_ids.size(1)
    max_length = model.cfg.block_size
    nlls = []
    prev_end_loc = 0
    
    for begin_loc in tqdm(range(0, seq_len, stride), desc="Evaluating"):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc 
        
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100
        
        outputs, _ = model(input_ids)
        shift_logits = outputs[0, :-1, :].contiguous()
        shift_labels = target_ids[0, 1:].contiguous()
        
        loss = F.cross_entropy(shift_logits, shift_labels, ignore_index=-100)
        nlls.append(loss)
        
        prev_end_loc = end_loc
        if end_loc == seq_len:
            break
            
    ppl = torch.exp(torch.stack(nlls).mean())
    print(f"Validation Perplexity: {ppl.item():.2f}")

@torch.no_grad()
def generate_samples(model, tokenizer, device, prompts, max_new_tokens=50, temperature=0.8):
    model.eval()
    model.to(device)
    print(f"\n--- Qualitative Generation ({device.upper()}) ---")
    
    for prompt in prompts:
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
        for _ in range(max_new_tokens):
            logits, _ = model(input_ids)
            next_token_logits = logits[:, -1, :] / temperature
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=-1)
            
        generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
        print(f"Prompt: {prompt}")
        print(f"Output: {generated_text}\n")

def main():
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    cfg = MambaLMConfig(
        vocab_size=tokenizer.vocab_size, 
        block_size=512,
        d_model=384, 
        n_layer=16,
        use_grad_checkpoint=False
    )
    
    model = MambaLM(cfg)
    ckpt_path = "/kaggle/input/models/ojasgupta074927/wikitext103-mamba-1-10500-32batch-size/pytorch/default/1/latest.pt"
    
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location="cpu")
        state_dict = ckpt["model_state"]
        if list(state_dict.keys())[0].startswith("module."):
            state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(state_dict)
        print(f"Loaded checkpoint from step {ckpt.get('step', 'unknown')}")
    else:
        print("Checkpoint not found. Running benchmarks on randomly initialized weights.")

    devices = ["cpu"]
    if torch.cuda.is_available():
        devices.append("cuda")

    test_prompts = [
        "The history of machine learning began with",
        "In a surprising turn of events, the global economy",
    ]

    for device in devices:
        benchmark_inference_speed_and_memory(model, device)
        generate_samples(model, tokenizer, device, test_prompts)
        calculate_validation_perplexity(model, tokenizer, device, stride=256)

if __name__ == "__main__":
    main()

Loaded checkpoint from step 10500

--- Benchmarking Speed & Memory (CPU) ---
Seq Len    | Latency (ms)    | Throughput (tok/s)   | Peak Memory (MB)
-----------------------------------------------------------------
128        | 361.25          | 354.32               | 2973.54        
256        | 740.50          | 345.71               | 3054.26        
512        | 1451.74         | 352.68               | 3103.73        
1024       | 3093.83         | 330.98               | 3218.11        

--- Qualitative Generation (CPU) ---
Prompt: The history of machine learning began with
Output: The history of machine learning began with the time . his skum . a proclaimingr an wide @ early the @-@ from ends . 30 , an an oct of VI ( unusually , being the time , which theb , a two of a creed . here , only . to

Prompt: In a surprising turn of events, the global economy
Output: In a surprising turn of events, the global economy , a the amount . that @ it musical . . the a a a dramatic of a major war 

Token indices sequence length is longer than the specified maximum sequence length for this model (251048 > 1024). Running this sequence through the model will result in indexing errors
Evaluating:  12%|█▏        | 122/981 [02:58<20:56,  1.46s/it]


KeyboardInterrupt: 

# GLUE and SuperGLUE

In [6]:
import os
import time
import math
import psutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer

try:
    from mamba_ssm import Mamba as _CudaMamba
    HAS_CUDA_MAMBA = True
except ImportError:
    HAS_CUDA_MAMBA = False

class PureTorchSelectiveScan(nn.Module):
    """Unfused, pure-PyTorch selective SSM scan. Correctness-check only."""
    def __init__(self, d_inner: int, d_state: int = 16):
        super().__init__()
        self.d_inner, self.d_state = d_inner, d_state
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_inner, 1)))
        self.D = nn.Parameter(torch.ones(d_inner))
        self.x_proj = nn.Linear(d_inner, d_state * 2 + d_inner, bias=False)
        self.dt_proj = nn.Linear(d_inner, d_inner, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B_, L, D_ = x.shape
        A = -torch.exp(self.A_log.float())
        proj = self.x_proj(x)
        Bmat, Cmat, dt_in = torch.split(proj, [self.d_state, self.d_state, self.d_inner], dim=-1)
        delta = F.softplus(self.dt_proj(dt_in))
        h = torch.zeros(B_, D_, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dA = torch.exp(delta[:, t].unsqueeze(-1) * A)
            dBx = delta[:, t].unsqueeze(-1) * Bmat[:, t].unsqueeze(1) * x[:, t].unsqueeze(-1)
            h = dA * h + dBx
            ys.append((h * Cmat[:, t].unsqueeze(1)).sum(-1) + self.D * x[:, t])
        return torch.stack(ys, dim=1)

class MambaBlock(nn.Module):
    """Causal Mamba block, pre-norm + residual."""
    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2, use_grad_checkpoint: bool = True):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.use_grad_checkpoint = use_grad_checkpoint
        d_inner = expand * d_model
        if HAS_CUDA_MAMBA:
            self.mamba = _CudaMamba(d_model=d_model, d_state=d_state, expand=expand)
        else:
            self.in_proj = nn.Linear(d_model, d_inner)
            self.out_proj = nn.Linear(d_inner, d_model)
            self.scan = PureTorchSelectiveScan(d_inner, d_state)

    def _forward_impl(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        if HAS_CUDA_MAMBA:
            out = self.mamba(h)
        else:
            out = self.out_proj(self.scan(self.in_proj(h)))
        return x + out

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_grad_checkpoint and self.training:
            return torch.utils.checkpoint.checkpoint(self._forward_impl, x, use_reentrant=False)
        return self._forward_impl(x)

@dataclass
class MambaLMConfig:
    vocab_size: int = 50257
    block_size: int = 1024
    d_model: int = 384
    n_layer: int = 16
    d_state: int = 16
    expand: int = 2
    use_grad_checkpoint: bool = True

class MambaLM(nn.Module):
    def __init__(self, cfg: MambaLMConfig):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([
            MambaBlock(cfg.d_model, d_state=cfg.d_state, expand=cfg.expand,
                       use_grad_checkpoint=cfg.use_grad_checkpoint)
            for _ in range(cfg.n_layer)
        ])
        self.norm_out = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight

        nn.init.normal_(self.embed.weight, mean=0.0, std=0.02)

        if not HAS_CUDA_MAMBA:
            self.apply(self._init_weights)
            for name, p in self.named_parameters():
                if name.endswith("out_proj.weight"):
                    nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None):
        x = self.embed(input_ids)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm_out(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1),
                ignore_index=-100,
            )
        return logits, loss

def get_memory_usage(device: str):
    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        process = psutil.Process(os.getpid())
        return process.memory_info().rss / (1024 ** 2)

@torch.no_grad()
def benchmark_inference_speed_and_memory(model, device, seq_lengths=[128, 256, 512, 1024], batch_size=1, warmup=3, runs=10):
    model.eval()
    model.to(device)
    
    print(f"\n--- Benchmarking Speed & Memory ({device.upper()}) ---")
    print(f"{'Seq Len':<10} | {'Latency (ms)':<15} | {'Throughput (tok/s)':<20} | {'Peak Memory (MB)':<15}")
    print("-" * 65)

    for seq_len in seq_lengths:
        input_ids = torch.randint(0, model.cfg.vocab_size, (batch_size, seq_len), device=device)
        
        if device == "cuda":
            torch.cuda.reset_peak_memory_stats()
        
        for _ in range(warmup):
            _ = model(input_ids)
            if device == "cuda": torch.cuda.synchronize()

        start_time = time.time()
        for _ in range(runs):
            _ = model(input_ids)
            if device == "cuda": torch.cuda.synchronize()
        end_time = time.time()

        avg_latency = (end_time - start_time) / runs
        throughput = (batch_size * seq_len) / avg_latency
        mem_usage = get_memory_usage(device)

        print(f"{seq_len:<10} | {avg_latency * 1000:<15.2f} | {throughput:<20.2f} | {mem_usage:<15.2f}")

@torch.no_grad()
def evaluate_zero_shot_benchmark(model, tokenizer, device, dataset_path, dataset_name, split="validation", limit=None):
    """
    Evaluates the model on classification datasets using a zero-shot prompting strategy.
    It compares the logit scores of the target label words to determine the prediction.
    """
    model.eval()
    model.to(device)
    
    print(f"\n--- Evaluating {dataset_path.upper()} ({dataset_name}) on {device.upper()} ---")
    dataset = load_dataset(dataset_path, dataset_name, split=split)
    if limit:
        dataset = dataset.select(range(min(limit, len(dataset))))

    correct = 0
    total = len(dataset)

    # Task-specific configurations
    if dataset_path == "glue" and dataset_name == "sst2":
        # SST-2: Sentiment Analysis (0: negative, 1: positive)
        label_map = {0: " negative", 1: " positive"}
        def create_prompt(example):
            return f"Review: {example['sentence']}\nSentiment:"
            
    elif dataset_path == "super_glue" and dataset_name == "boolq":
        # BoolQ: Question Answering (0: False, 1: True)
        label_map = {0: " False", 1: " True"}
        def create_prompt(example):
            return f"Passage: {example['passage']}\nQuestion: {example['question']}\nAnswer:"
    else:
        raise ValueError("Unsupported benchmark configuration.")

    # Get token IDs for the exact target words
    target_token_ids = {
        label_idx: tokenizer.encode(word, add_special_tokens=False)[0] 
        for label_idx, word in label_map.items()
    }

    for example in tqdm(dataset, desc=f"Scoring {dataset_name}"):
        prompt = create_prompt(example)
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
        
        logits, _ = model(input_ids)
        next_token_logits = logits[0, -1, :] # Logits for the immediate next token
        
        # Extract the probability scores for our specific label choices
        label_scores = {
            label_idx: next_token_logits[token_id].item() 
            for label_idx, token_id in target_token_ids.items()
        }
        
        # The model predicts the label with the highest logit score
        prediction = max(label_scores, key=label_scores.get)
        
        if prediction == example["label"]:
            correct += 1

    accuracy = (correct / total) * 100
    print(f"{dataset_name.upper()} Zero-Shot Accuracy: {accuracy:.2f}% ({correct}/{total})")
    return accuracy

def main():
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    cfg = MambaLMConfig(
        vocab_size=tokenizer.vocab_size, 
        block_size=512,
        d_model=384, 
        n_layer=16,
        use_grad_checkpoint=False
    )
    
    model = MambaLM(cfg)
    ckpt_path = "/kaggle/input/models/ojasgupta074927/wikitext103-mamba-1-10500-32batch-size/pytorch/default/1/latest.pt"
    
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location="cpu")
        state_dict = ckpt["model_state"]
        if list(state_dict.keys())[0].startswith("module."):
            state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(state_dict)
        print(f"Loaded checkpoint from step {ckpt.get('step', 'unknown')}")
    else:
        print("Checkpoint not found. Running benchmarks on randomly initialized weights.")

    devices = ["cpu"]
    if torch.cuda.is_available():
        devices.append("cuda")

    for device in devices:
        if device == "cpu" and HAS_CUDA_MAMBA:
            print(f"\n{'='*65}\n[NOTICE] Skipping CPU benchmarks.")
            print("The model is using the fused `mamba_ssm` backend. Its underlying ")
            print("C++ kernels (`causal_conv1d_cuda` and `selective_scan_fn`) are ")
            print("strictly written for GPU and will throw a RuntimeError on CPU.")
            print(f"{'='*65}")
            continue

        benchmark_inference_speed_and_memory(model, device)
        
        # GLUE: SST-2 (Sentiment Analysis)
        evaluate_zero_shot_benchmark(model, tokenizer, device, "glue", "sst2")
        
        # SuperGLUE: BoolQ (Boolean Question Answering)
        # Limiting to 1000 examples as BoolQ passages are long and computationally heavy
        evaluate_zero_shot_benchmark(model, tokenizer, device, "super_glue", "boolq", limit=1000)

if __name__ == "__main__":
    main()

Loaded checkpoint from step 10500

[NOTICE] Skipping CPU benchmarks.
The model is using the fused `mamba_ssm` backend. Its underlying 
C++ kernels (`causal_conv1d_cuda` and `selective_scan_fn`) are 
strictly written for GPU and will throw a RuntimeError on CPU.

--- Benchmarking Speed & Memory (CUDA) ---
Seq Len    | Latency (ms)    | Throughput (tok/s)   | Peak Memory (MB)
-----------------------------------------------------------------
128        | 13.87           | 9227.23              | 168.11         
256        | 15.21           | 16829.52             | 193.75         
512        | 18.80           | 27235.50             | 242.29         
1024       | 26.73           | 38314.60             | 341.20         

--- Evaluating GLUE (sst2) on CUDA ---


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

... (9 outputs left out)

Generating validation split:   0%|          | 0/3270 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3245 [00:00<?, ? examples/s]

Scoring boolq: 100%|██████████| 1000/1000 [00:15<00:00, 63.98it/s]


BOOLQ Zero-Shot Accuracy: 59.30% (593/1000)


# Supervised fine tuning for GLUE and SuperGLUE

In [7]:
import os
import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

try:
    from mamba_ssm import Mamba as _CudaMamba
    HAS_CUDA_MAMBA = True
except ImportError:
    HAS_CUDA_MAMBA = False

# -----------------------------------------------------------------------
# 1. Base Architecture Components
# -----------------------------------------------------------------------

class PureTorchSelectiveScan(nn.Module):
    def __init__(self, d_inner: int, d_state: int = 16):
        super().__init__()
        self.d_inner, self.d_state = d_inner, d_state
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_inner, 1)))
        self.D = nn.Parameter(torch.ones(d_inner))
        self.x_proj = nn.Linear(d_inner, d_state * 2 + d_inner, bias=False)
        self.dt_proj = nn.Linear(d_inner, d_inner, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B_, L, D_ = x.shape
        A = -torch.exp(self.A_log.float())
        proj = self.x_proj(x)
        Bmat, Cmat, dt_in = torch.split(proj, [self.d_state, self.d_state, self.d_inner], dim=-1)
        delta = F.softplus(self.dt_proj(dt_in))
        h = torch.zeros(B_, D_, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dA = torch.exp(delta[:, t].unsqueeze(-1) * A)
            dBx = delta[:, t].unsqueeze(-1) * Bmat[:, t].unsqueeze(1) * x[:, t].unsqueeze(-1)
            h = dA * h + dBx
            ys.append((h * Cmat[:, t].unsqueeze(1)).sum(-1) + self.D * x[:, t])
        return torch.stack(ys, dim=1)

def mamba_pure_pytorch_forward(mamba_module, x):
    B, L, D = x.shape
    xz = mamba_module.in_proj(x) 
    x_in, z = xz.chunk(2, dim=-1) 
    x_in = x_in.transpose(1, 2)
    x_in = F.pad(x_in, (mamba_module.d_conv - 1, 0))
    x_in = mamba_module.conv1d(x_in)
    x_in = x_in.transpose(1, 2) 
    x_in = F.silu(x_in)
    x_proj = mamba_module.x_proj(x_in)
    dt, B_mat, C_mat = torch.split(x_proj, [mamba_module.dt_rank, mamba_module.d_state, mamba_module.d_state], dim=-1)
    dt = F.softplus(mamba_module.dt_proj(dt)) 
    A = -torch.exp(mamba_module.A_log.float()) 
    D_mat = mamba_module.D.float() 
    h = torch.zeros(B, mamba_module.d_inner, mamba_module.d_state, device=x.device, dtype=x.dtype)
    ys = []
    for t in range(L):
        dt_t = dt[:, t].unsqueeze(-1)
        dA = torch.exp(dt_t * A)
        dBx = dt_t * B_mat[:, t].unsqueeze(1) * x_in[:, t].unsqueeze(-1)
        h = dA * h + dBx
        y_t = (h * C_mat[:, t].unsqueeze(1)).sum(dim=-1) + D_mat * x_in[:, t]
        ys.append(y_t)
    y = torch.stack(ys, dim=1)
    out = y * F.silu(z)
    return mamba_module.out_proj(out)

class MambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2, use_grad_checkpoint: bool = True):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.use_grad_checkpoint = use_grad_checkpoint
        d_inner = expand * d_model
        if HAS_CUDA_MAMBA:
            self.mamba = _CudaMamba(d_model=d_model, d_state=d_state, expand=expand)
        else:
            self.in_proj = nn.Linear(d_model, d_inner)
            self.out_proj = nn.Linear(d_inner, d_model)
            self.scan = PureTorchSelectiveScan(d_inner, d_state)

    def _forward_impl(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        if HAS_CUDA_MAMBA:
            if x.is_cuda:
                out = self.mamba(h)
            else:
                out = mamba_pure_pytorch_forward(self.mamba, h)
        else:
            out = self.out_proj(self.scan(self.in_proj(h)))
        return x + out

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_grad_checkpoint and self.training:
            return torch.utils.checkpoint.checkpoint(self._forward_impl, x, use_reentrant=False)
        return self._forward_impl(x)

@dataclass
class MambaConfig:
    vocab_size: int = 50257
    d_model: int = 384
    n_layer: int = 16
    d_state: int = 16
    expand: int = 2
    use_grad_checkpoint: bool = False

# -----------------------------------------------------------------------
# 2. Classification Model
# -----------------------------------------------------------------------

class MambaForSequenceClassification(nn.Module):
    def __init__(self, cfg: MambaConfig, num_labels: int = 2):
        super().__init__()
        self.cfg = cfg
        self.num_labels = num_labels
        
        # Base Model
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([
            MambaBlock(cfg.d_model, cfg.d_state, cfg.expand, cfg.use_grad_checkpoint)
            for _ in range(cfg.n_layer)
        ])
        self.norm_out = nn.LayerNorm(cfg.d_model)
        
        # Classification Head
        self.classifier = nn.Linear(cfg.d_model, num_labels, bias=False)

        # Init classification head
        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, labels: Optional[torch.Tensor] = None):
        x = self.embed(input_ids)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm_out(x)

        # Extract the hidden state of the last non-padded token
        # sequence_lengths tracks the index of the last token (sum of mask - 1)
        sequence_lengths = attention_mask.sum(dim=1) - 1
        sequence_lengths = sequence_lengths.clamp(min=0)
        
        # Gather the pooled state: shape (batch_size, d_model)
        pooled_logits = x[torch.arange(x.shape[0], device=x.device), sequence_lengths]
        
        logits = self.classifier(pooled_logits)
        
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
            
        return logits, loss

# -----------------------------------------------------------------------
# 3. Fine-Tuning Pipeline
# -----------------------------------------------------------------------

def prepare_sst2_dataloaders(tokenizer, batch_size=32, max_length=128):
    print("Loading and tokenizing SST-2 dataset...")
    dataset = load_dataset("glue", "sst2")
    
    def tokenize_function(examples):
        return tokenizer(
            examples["sentence"], 
            padding="max_length", 
            truncation=True, 
            max_length=max_length
        )

    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])

    train_loader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=batch_size)
    eval_loader = DataLoader(tokenized_datasets["validation"], batch_size=batch_size)
    
    return train_loader, eval_loader

@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits, loss = model(input_ids, attention_mask, labels=labels)
        total_loss += loss.item()
        
        predictions = torch.argmax(logits, dim=-1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    accuracy = correct / total
    avg_loss = total_loss / len(dataloader)
    return accuracy, avg_loss

def train_classifier(model, train_loader, eval_loader, device, epochs=3, lr=2e-5):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    total_steps = len(train_loader) * epochs
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.1
    )

    print(f"\n--- Starting Fine-Tuning on {device.upper()} ---")
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch in progress_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            logits, loss = model(input_ids, attention_mask, labels=labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            optimizer.step()
            scheduler.step()
            
            total_train_loss += loss.item()
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        avg_train_loss = total_train_loss / len(train_loader)
        val_acc, val_loss = evaluate(model, eval_loader, device)
        
        print(f"Epoch {epoch+1} Summary:")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")
        print(f"  Val Acc:    {val_acc * 100:.2f}%\n")
        
    return model

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    cfg = MambaConfig()
    model = MambaForSequenceClassification(cfg, num_labels=2)
    
    # Load base weights, strict=False to ignore missing 'classifier' and unexpected 'lm_head'
    ckpt_path = "/kaggle/input/models/ojasgupta074927/wikitext103-mamba-1-10500-32batch-size/pytorch/default/1/latest.pt"
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location="cpu")
        state_dict = ckpt["model_state"]
        
        # Clean DataParallel prefixes if they exist
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        
        # Filter out the lm_head as we don't need it for classification
        base_state_dict = {k: v for k, v in state_dict.items() if "lm_head" not in k}
        
        missing, unexpected = model.load_state_dict(base_state_dict, strict=False)
        print("Loaded base pre-trained weights.")
    else:
        print("Checkpoint not found. Initializing with random weights.")

    train_loader, eval_loader = prepare_sst2_dataloaders(tokenizer, batch_size=64, max_length=128)
    
    # Train (3 epochs, smaller learning rate for fine-tuning)
    train_classifier(model, train_loader, eval_loader, device, epochs=3, lr=3e-5)

if __name__ == "__main__":
    main()

Loaded base pre-trained weights.
Loading and tokenizing SST-2 dataset...


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]


--- Starting Fine-Tuning on CUDA ---


Epoch 1/3: 100%|██████████| 1053/1053 [06:36<00:00,  2.65it/s, loss=0.2611]


Epoch 1 Summary:
  Train Loss: 0.3969
  Val Loss:   0.3401
  Val Acc:    86.35%



Epoch 2/3: 100%|██████████| 1053/1053 [06:35<00:00,  2.66it/s, loss=0.1628]


Epoch 2 Summary:
  Train Loss: 0.1525
  Val Loss:   0.3738
  Val Acc:    86.93%



Epoch 3/3: 100%|██████████| 1053/1053 [06:35<00:00,  2.66it/s, loss=0.0430]


Epoch 3 Summary:
  Train Loss: 0.0655
  Val Loss:   0.4897
  Val Acc:    86.01%



In [8]:
import os
import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

try:
    from mamba_ssm import Mamba as _CudaMamba
    HAS_CUDA_MAMBA = True
except ImportError:
    HAS_CUDA_MAMBA = False

# -----------------------------------------------------------------------
# 1. Base Architecture Components
# -----------------------------------------------------------------------

class PureTorchSelectiveScan(nn.Module):
    def __init__(self, d_inner: int, d_state: int = 16):
        super().__init__()
        self.d_inner, self.d_state = d_inner, d_state
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32).repeat(d_inner, 1)))
        self.D = nn.Parameter(torch.ones(d_inner))
        self.x_proj = nn.Linear(d_inner, d_state * 2 + d_inner, bias=False)
        self.dt_proj = nn.Linear(d_inner, d_inner, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B_, L, D_ = x.shape
        A = -torch.exp(self.A_log.float())
        proj = self.x_proj(x)
        Bmat, Cmat, dt_in = torch.split(proj, [self.d_state, self.d_state, self.d_inner], dim=-1)
        delta = F.softplus(self.dt_proj(dt_in))
        h = torch.zeros(B_, D_, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dA = torch.exp(delta[:, t].unsqueeze(-1) * A)
            dBx = delta[:, t].unsqueeze(-1) * Bmat[:, t].unsqueeze(1) * x[:, t].unsqueeze(-1)
            h = dA * h + dBx
            ys.append((h * Cmat[:, t].unsqueeze(1)).sum(-1) + self.D * x[:, t])
        return torch.stack(ys, dim=1)

def mamba_pure_pytorch_forward(mamba_module, x):
    B, L, D = x.shape
    xz = mamba_module.in_proj(x) 
    x_in, z = xz.chunk(2, dim=-1) 
    x_in = x_in.transpose(1, 2)
    x_in = F.pad(x_in, (mamba_module.d_conv - 1, 0))
    x_in = mamba_module.conv1d(x_in)
    x_in = x_in.transpose(1, 2) 
    x_in = F.silu(x_in)
    x_proj = mamba_module.x_proj(x_in)
    dt, B_mat, C_mat = torch.split(x_proj, [mamba_module.dt_rank, mamba_module.d_state, mamba_module.d_state], dim=-1)
    dt = F.softplus(mamba_module.dt_proj(dt)) 
    A = -torch.exp(mamba_module.A_log.float()) 
    D_mat = mamba_module.D.float() 
    h = torch.zeros(B, mamba_module.d_inner, mamba_module.d_state, device=x.device, dtype=x.dtype)
    ys = []
    for t in range(L):
        dt_t = dt[:, t].unsqueeze(-1)
        dA = torch.exp(dt_t * A)
        dBx = dt_t * B_mat[:, t].unsqueeze(1) * x_in[:, t].unsqueeze(-1)
        h = dA * h + dBx
        y_t = (h * C_mat[:, t].unsqueeze(1)).sum(dim=-1) + D_mat * x_in[:, t]
        ys.append(y_t)
    y = torch.stack(ys, dim=1)
    out = y * F.silu(z)
    return mamba_module.out_proj(out)

class MambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2, use_grad_checkpoint: bool = True):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.use_grad_checkpoint = use_grad_checkpoint
        d_inner = expand * d_model
        if HAS_CUDA_MAMBA:
            self.mamba = _CudaMamba(d_model=d_model, d_state=d_state, expand=expand)
        else:
            self.in_proj = nn.Linear(d_model, d_inner)
            self.out_proj = nn.Linear(d_inner, d_model)
            self.scan = PureTorchSelectiveScan(d_inner, d_state)

    def _forward_impl(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        if HAS_CUDA_MAMBA:
            if x.is_cuda:
                out = self.mamba(h)
            else:
                out = mamba_pure_pytorch_forward(self.mamba, h)
        else:
            out = self.out_proj(self.scan(self.in_proj(h)))
        return x + out

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_grad_checkpoint and self.training:
            return torch.utils.checkpoint.checkpoint(self._forward_impl, x, use_reentrant=False)
        return self._forward_impl(x)

@dataclass
class MambaConfig:
    vocab_size: int = 50257
    d_model: int = 384
    n_layer: int = 16
    d_state: int = 16
    expand: int = 2
    use_grad_checkpoint: bool = False

# -----------------------------------------------------------------------
# 2. Classification Model
# -----------------------------------------------------------------------

class MambaForSequenceClassification(nn.Module):
    def __init__(self, cfg: MambaConfig, num_labels: int = 2):
        super().__init__()
        self.cfg = cfg
        self.num_labels = num_labels
        
        # Base Model
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([
            MambaBlock(cfg.d_model, cfg.d_state, cfg.expand, cfg.use_grad_checkpoint)
            for _ in range(cfg.n_layer)
        ])
        self.norm_out = nn.LayerNorm(cfg.d_model)
        
        # Classification Head
        self.classifier = nn.Linear(cfg.d_model, num_labels, bias=False)

        # Init classification head
        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, labels: Optional[torch.Tensor] = None):
        x = self.embed(input_ids)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm_out(x)

        # Extract the hidden state of the last non-padded token
        sequence_lengths = attention_mask.sum(dim=1) - 1
        sequence_lengths = sequence_lengths.clamp(min=0)
        
        # Gather the pooled state: shape (batch_size, d_model)
        pooled_logits = x[torch.arange(x.shape[0], device=x.device), sequence_lengths]
        
        logits = self.classifier(pooled_logits)
        
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
            
        return logits, loss

# -----------------------------------------------------------------------
# 3. Fine-Tuning Pipeline
# -----------------------------------------------------------------------

def prepare_sst2_dataloaders(tokenizer, batch_size=32, max_length=128):
    print("Loading and tokenizing SST-2 dataset...")
    dataset = load_dataset("glue", "sst2")
    
    def tokenize_function(examples):
        return tokenizer(
            examples["sentence"], 
            padding="max_length", 
            truncation=True, 
            max_length=max_length
        )

    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])

    train_loader = DataLoader(tokenized_datasets["train"], shuffle=True, batch_size=batch_size)
    eval_loader = DataLoader(tokenized_datasets["validation"], batch_size=batch_size)
    
    return train_loader, eval_loader

@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits, loss = model(input_ids, attention_mask, labels=labels)
        total_loss += loss.item()
        
        predictions = torch.argmax(logits, dim=-1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    accuracy = correct / total
    avg_loss = total_loss / len(dataloader)
    return accuracy, avg_loss

def train_classifier(model, train_loader, eval_loader, device, epochs=3, lr=2e-5, output_dir="./finetuned_models"):
    os.makedirs(output_dir, exist_ok=True)
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    total_steps = len(train_loader) * epochs
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.1
    )

    print(f"\n--- Starting Fine-Tuning on {device.upper()} ---")
    
    best_val_acc = 0.0

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch in progress_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            logits, loss = model(input_ids, attention_mask, labels=labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            optimizer.step()
            scheduler.step()
            
            total_train_loss += loss.item()
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
            
        avg_train_loss = total_train_loss / len(train_loader)
        val_acc, val_loss = evaluate(model, eval_loader, device)
        
        print(f"Epoch {epoch+1} Summary:")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")
        print(f"  Val Acc:    {val_acc * 100:.2f}%")
        
        # Save the best model based on validation accuracy
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_path = os.path.join(output_dir, "best_model.pt")
            torch.save(model.state_dict(), save_path)
            print(f"  --> New best validation accuracy! Saved to {save_path}\n")
        else:
            print("\n")

    # Save the final state at the end of all epochs just in case
    final_path = os.path.join(output_dir, "latest_model.pt")
    torch.save(model.state_dict(), final_path)
    print(f"Training complete. Final state saved to {final_path}")
    
    return model

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    cfg = MambaConfig()
    model = MambaForSequenceClassification(cfg, num_labels=2)
    
    # Load base weights, strict=False to ignore missing 'classifier' and unexpected 'lm_head'
    ckpt_path = "/kaggle/input/models/ojasgupta074927/wikitext103-mamba-1-10500-32batch-size/pytorch/default/1/latest.pt"
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location="cpu")
        state_dict = ckpt["model_state"]
        
        # Clean DataParallel prefixes if they exist
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        
        # Filter out the lm_head as we don't need it for classification
        base_state_dict = {k: v for k, v in state_dict.items() if "lm_head" not in k}
        
        missing, unexpected = model.load_state_dict(base_state_dict, strict=False)
        print("Loaded base pre-trained weights.")
    else:
        print("Checkpoint not found. Initializing with random weights.")

    train_loader, eval_loader = prepare_sst2_dataloaders(tokenizer, batch_size=64, max_length=128)
    
    # Output directory for saving checkpoints
    output_dir = "/kaggle/working/finetuned_checkpoints"
    
    # Train (3 epochs, smaller learning rate for fine-tuning)
    train_classifier(model, train_loader, eval_loader, device, epochs=3, lr=3e-5, output_dir=output_dir)

if __name__ == "__main__":
    main()

Loaded base pre-trained weights.
Loading and tokenizing SST-2 dataset...

--- Starting Fine-Tuning on CUDA ---


Epoch 1/3: 100%|██████████| 1053/1053 [06:36<00:00,  2.65it/s, loss=0.3715]


Epoch 1 Summary:
  Train Loss: 0.4017
  Val Loss:   0.3386
  Val Acc:    86.58%
  --> New best validation accuracy! Saved to /kaggle/working/finetuned_checkpoints/best_model.pt



Epoch 2/3: 100%|██████████| 1053/1053 [06:35<00:00,  2.66it/s, loss=0.1155]


Epoch 2 Summary:
  Train Loss: 0.1535
  Val Loss:   0.3611
  Val Acc:    86.81%
  --> New best validation accuracy! Saved to /kaggle/working/finetuned_checkpoints/best_model.pt



Epoch 3/3: 100%|██████████| 1053/1053 [06:35<00:00,  2.66it/s, loss=0.1084]


Epoch 3 Summary:
  Train Loss: 0.0672
  Val Loss:   0.4811
  Val Acc:    86.01%


Training complete. Final state saved to /kaggle/working/finetuned_checkpoints/latest_model.pt


In [12]:
import shutil

src = "/kaggle/working/finetuned_checkpoints/latest_model.pt"
dst = "/kaggle/working/latest_model.pt"

shutil.copy2(src, dst)

print(f"Copied to: {dst}")

Copied to: /kaggle/working/latest_model.pt


In [2]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
from tqdm import tqdm

try:
    from mamba_ssm import Mamba as _CudaMamba
    HAS_CUDA_MAMBA = True
except ImportError:
    HAS_CUDA_MAMBA = False

# -----------------------------------------------------------------------
# 1. Unified Architecture (Matches Checkpoint Keys 1:1)
# -----------------------------------------------------------------------

class TorchMamba(nn.Module):
    """Pure PyTorch implementation of Mamba with identical state_dict structure to mamba_ssm.Mamba."""
    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = int(self.expand * self.d_model)
        self.dt_rank = math.ceil(self.d_model / 16)

        self.in_proj = nn.Linear(self.d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            bias=True,
            kernel_size=d_conv,
            groups=self.d_inner,
            padding=d_conv - 1,
        )
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + self.d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, D = x.shape
        xz = self.in_proj(x)
        x_in, z = xz.chunk(2, dim=-1)

        # 1D Causal Convolution
        x_in = x_in.transpose(1, 2)
        x_in = self.conv1d(x_in)[:, :, :L]
        x_in = x_in.transpose(1, 2)
        x_in = F.silu(x_in)

        # Project dt, B, C
        x_dbl = self.x_proj(x_in)
        dt, B_mat, C_mat = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt))

        A = -torch.exp(self.A_log.float())
        D_mat = self.D.float()

        # Sequential scan
        h = torch.zeros(B, self.d_inner, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dt_t = dt[:, t].unsqueeze(-1)
            dA = torch.exp(dt_t * A)
            dBx = dt_t * B_mat[:, t].unsqueeze(1) * x_in[:, t].unsqueeze(-1)
            h = dA * h + dBx
            y_t = (h * C_mat[:, t].unsqueeze(1)).sum(dim=-1) + D_mat * x_in[:, t]
            ys.append(y_t)

        y = torch.stack(ys, dim=1)
        out = y * F.silu(z)
        return self.out_proj(out)


class MambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        if HAS_CUDA_MAMBA:
            self.mamba = _CudaMamba(d_model=d_model, d_state=d_state, expand=expand)
        else:
            self.mamba = TorchMamba(d_model=d_model, d_state=d_state, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        if HAS_CUDA_MAMBA and not x.is_cuda:
            # If tensor is on CPU but CUDA Mamba object exists, fallback via pure-torch ops
            out = self._fallback_cpu_forward(h)
        else:
            out = self.mamba(h)
        return x + out

    def _fallback_cpu_forward(self, x: torch.Tensor) -> torch.Tensor:
        m = self.mamba
        B, L, D = x.shape
        xz = m.in_proj(x)
        x_in, z = xz.chunk(2, dim=-1)
        x_in = x_in.transpose(1, 2)
        x_in = F.pad(x_in, (m.d_conv - 1, 0))
        x_in = m.conv1d(x_in)[:, :, :L]
        x_in = x_in.transpose(1, 2)
        x_in = F.silu(x_in)

        x_proj = m.x_proj(x_in)
        dt, B_mat, C_mat = torch.split(x_proj, [m.dt_rank, m.d_state, m.d_state], dim=-1)
        dt = F.softplus(m.dt_proj(dt))

        A = -torch.exp(m.A_log.float())
        D_mat = m.D.float()
        h = torch.zeros(B, m.d_inner, m.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dt_t = dt[:, t].unsqueeze(-1)
            dA = torch.exp(dt_t * A)
            dBx = dt_t * B_mat[:, t].unsqueeze(1) * x_in[:, t].unsqueeze(-1)
            h = dA * h + dBx
            y_t = (h * C_mat[:, t].unsqueeze(1)).sum(dim=-1) + D_mat * x_in[:, t]
            ys.append(y_t)
        y = torch.stack(ys, dim=1)
        return m.out_proj(y * F.silu(z))


@dataclass
class MambaConfig:
    vocab_size: int = 50257
    d_model: int = 384
    n_layer: int = 16
    d_state: int = 16
    expand: int = 2


class MambaForSequenceClassification(nn.Module):
    def __init__(self, cfg: MambaConfig, num_labels: int = 2):
        super().__init__()
        self.cfg = cfg
        self.num_labels = num_labels
        
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([
            MambaBlock(cfg.d_model, cfg.d_state, cfg.expand)
            for _ in range(cfg.n_layer)
        ])
        self.norm_out = nn.LayerNorm(cfg.d_model)
        self.classifier = nn.Linear(cfg.d_model, num_labels, bias=False)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        x = self.embed(input_ids)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm_out(x)

        # Extract hidden state at last non-padded token index
        sequence_lengths = (attention_mask.sum(dim=1) - 1).clamp(min=0)
        pooled_logits = x[torch.arange(x.shape[0], device=x.device), sequence_lengths]
        logits = self.classifier(pooled_logits)
        return logits

# -----------------------------------------------------------------------
# 2. Evaluation Dataloaders
# -----------------------------------------------------------------------

def prepare_sst2_eval_loader(tokenizer, batch_size=64, max_length=128):
    print("Preparing SST-2 validation set...")
    dataset = load_dataset("glue", "sst2", split="validation")
    
    def tokenize_function(examples):
        return tokenizer(
            examples["sentence"], 
            padding="max_length", 
            truncation=True, 
            max_length=max_length
        )

    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    return DataLoader(tokenized_dataset, batch_size=batch_size)


def prepare_boolq_eval_loader(tokenizer, batch_size=64, max_length=256):
    print("Preparing BoolQ validation set...")
    dataset = load_dataset("super_glue", "boolq", split="validation")
    
    def tokenize_function(examples):
        texts = [f"Question: {q} Passage: {p}" for q, p in zip(examples["question"], examples["passage"])]
        return tokenizer(
            texts, 
            padding="max_length", 
            truncation=True, 
            max_length=max_length
        )

    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    tokenized_dataset = tokenized_dataset.map(lambda x: {"label": int(x["label"])})
    tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    return DataLoader(tokenized_dataset, batch_size=batch_size)

# -----------------------------------------------------------------------
# 3. Main Evaluation Runner
# -----------------------------------------------------------------------

@torch.no_grad()
def evaluate_loader(model, dataloader, device, dataset_name):
    model.eval()
    model.to(device)
    correct = 0
    total = 0

    print(f"\n--- Running Evaluation on {dataset_name.upper()} ({device.upper()}) ---")
    for batch in tqdm(dataloader, desc=f"Evaluating {dataset_name}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, attention_mask)
        predictions = torch.argmax(logits, dim=-1)
        
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    accuracy = (correct / total) * 100
    print(f"-> {dataset_name.upper()} Accuracy: {accuracy:.2f}% ({correct}/{total})")
    return accuracy


def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    cfg = MambaConfig()
    model = MambaForSequenceClassification(cfg, num_labels=2)
    
    ckpt_path = "/kaggle/working/finetuned_checkpoints/best_model.pt"
    if not os.path.exists(ckpt_path):
        # Fallback search if path differs
        ckpt_path = "/kaggle/working/finetuned_models/best_model.pt"

    if os.path.exists(ckpt_path):
        state_dict = torch.load(ckpt_path, map_location="cpu")
        # Strip DataParallel/DDP module prefixes if present
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(state_dict, strict=True)
        print(f"Successfully loaded fine-tuned weights from {ckpt_path}")
    else:
        print(f"[Error] Checkpoint not found at {ckpt_path}. Please check output path.")
        return

    # 1. SST-2 (GLUE)
    sst2_loader = prepare_sst2_eval_loader(tokenizer, batch_size=64, max_length=128)
    evaluate_loader(model, sst2_loader, device, "SST-2 (GLUE)")
    
    # 2. BoolQ (SuperGLUE)
    boolq_loader = prepare_boolq_eval_loader(tokenizer, batch_size=64, max_length=256)
    evaluate_loader(model, boolq_loader, device, "BoolQ (SuperGLUE)")


if __name__ == "__main__":
    main()

Successfully loaded fine-tuned weights from /kaggle/working/finetuned_checkpoints/best_model.pt
Preparing SST-2 validation set...


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

... (14 outputs left out)

Map:   0%|          | 0/3270 [00:00<?, ? examples/s]


--- Running Evaluation on BOOLQ (SUPERGLUE) (CUDA) ---


Evaluating BoolQ (SuperGLUE): 100%|██████████| 52/52 [01:11<00:00,  1.37s/it]

-> BOOLQ (SUPERGLUE) Accuracy: 59.45% (1944/3270)


# Train on BOOLQ

In [3]:
import os
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

try:
    from mamba_ssm import Mamba as _CudaMamba
    HAS_CUDA_MAMBA = True
except ImportError:
    HAS_CUDA_MAMBA = False

# -----------------------------------------------------------------------
# 1. Unified Architecture Definition
# -----------------------------------------------------------------------

class TorchMamba(nn.Module):
    """Pure PyTorch fallback with identical parameter layout to mamba_ssm.Mamba."""
    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = int(self.expand * self.d_model)
        self.dt_rank = math.ceil(self.d_model / 16)

        self.in_proj = nn.Linear(self.d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            bias=True,
            kernel_size=d_conv,
            groups=self.d_inner,
            padding=d_conv - 1,
        )
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + self.d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, D = x.shape
        xz = self.in_proj(x)
        x_in, z = xz.chunk(2, dim=-1)

        x_in = x_in.transpose(1, 2)
        x_in = self.conv1d(x_in)[:, :, :L]
        x_in = x_in.transpose(1, 2)
        x_in = F.silu(x_in)

        x_dbl = self.x_proj(x_in)
        dt, B_mat, C_mat = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt))

        A = -torch.exp(self.A_log.float())
        D_mat = self.D.float()

        h = torch.zeros(B, self.d_inner, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dt_t = dt[:, t].unsqueeze(-1)
            dA = torch.exp(dt_t * A)
            dBx = dt_t * B_mat[:, t].unsqueeze(1) * x_in[:, t].unsqueeze(-1)
            h = dA * h + dBx
            y_t = (h * C_mat[:, t].unsqueeze(1)).sum(dim=-1) + D_mat * x_in[:, t]
            ys.append(y_t)

        y = torch.stack(ys, dim=1)
        out = y * F.silu(z)
        return self.out_proj(out)


class MambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        if HAS_CUDA_MAMBA:
            self.mamba = _CudaMamba(d_model=d_model, d_state=d_state, expand=expand)
        else:
            self.mamba = TorchMamba(d_model=d_model, d_state=d_state, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        if HAS_CUDA_MAMBA and not x.is_cuda:
            out = self._fallback_cpu_forward(h)
        else:
            out = self.mamba(h)
        return x + out

    def _fallback_cpu_forward(self, x: torch.Tensor) -> torch.Tensor:
        m = self.mamba
        B, L, D = x.shape
        xz = m.in_proj(x)
        x_in, z = xz.chunk(2, dim=-1)
        x_in = x_in.transpose(1, 2)
        x_in = F.pad(x_in, (m.d_conv - 1, 0))
        x_in = m.conv1d(x_in)[:, :, :L]
        x_in = x_in.transpose(1, 2)
        x_in = F.silu(x_in)

        x_proj = m.x_proj(x_in)
        dt, B_mat, C_mat = torch.split(x_proj, [m.dt_rank, m.d_state, m.d_state], dim=-1)
        dt = F.softplus(m.dt_proj(dt))

        A = -torch.exp(m.A_log.float())
        D_mat = m.D.float()
        h = torch.zeros(B, m.d_inner, m.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dt_t = dt[:, t].unsqueeze(-1)
            dA = torch.exp(dt_t * A)
            dBx = dt_t * B_mat[:, t].unsqueeze(1) * x_in[:, t].unsqueeze(-1)
            h = dA * h + dBx
            y_t = (h * C_mat[:, t].unsqueeze(1)).sum(dim=-1) + D_mat * x_in[:, t]
            ys.append(y_t)
        y = torch.stack(ys, dim=1)
        return m.out_proj(y * F.silu(z))


@dataclass
class MambaConfig:
    vocab_size: int = 50257
    d_model: int = 384
    n_layer: int = 16
    d_state: int = 16
    expand: int = 2


class MambaForSequenceClassification(nn.Module):
    def __init__(self, cfg: MambaConfig, num_labels: int = 2):
        super().__init__()
        self.cfg = cfg
        self.num_labels = num_labels

        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([
            MambaBlock(cfg.d_model, cfg.d_state, cfg.expand)
            for _ in range(cfg.n_layer)
        ])
        self.norm_out = nn.LayerNorm(cfg.d_model)
        self.classifier = nn.Linear(cfg.d_model, num_labels, bias=False)

        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, labels: Optional[torch.Tensor] = None):
        x = self.embed(input_ids)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm_out(x)

        sequence_lengths = (attention_mask.sum(dim=1) - 1).clamp(min=0)
        pooled_logits = x[torch.arange(x.shape[0], device=x.device), sequence_lengths]
        logits = self.classifier(pooled_logits)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return logits, loss

# -----------------------------------------------------------------------
# 2. BoolQ Dataset Pipeline & Fine-Tuning
# -----------------------------------------------------------------------

def prepare_boolq_dataloaders(tokenizer, batch_size=32, max_length=256):
    print("Loading and tokenizing BoolQ dataset (SuperGLUE)...")
    dataset = load_dataset("super_glue", "boolq")

    def tokenize_fn(examples):
        texts = [f"Question: {q} Passage: {p}" for q, p in zip(examples["question"], examples["passage"])]
        return tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=max_length
        )

    tokenized = dataset.map(tokenize_fn, batched=True)
    tokenized = tokenized.map(lambda x: {"label": int(x["label"])})
    tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

    train_loader = DataLoader(tokenized["train"], shuffle=True, batch_size=batch_size)
    val_loader = DataLoader(tokenized["validation"], batch_size=batch_size)
    return train_loader, val_loader


@torch.no_grad()
def evaluate_split(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits, loss = model(input_ids, attention_mask, labels=labels)
        if loss is not None:
            total_loss += loss.item()

        preds = torch.argmax(logits, dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    accuracy = (correct / total) * 100
    avg_loss = total_loss / max(1, len(dataloader))
    return accuracy, avg_loss


def train_boolq(model, train_loader, val_loader, device, epochs=3, lr=3e-5, save_dir="/kaggle/working/finetuned_checkpoints"):
    os.makedirs(save_dir, exist_ok=True)
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    total_steps = len(train_loader) * epochs
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.1
    )

    print(f"\n--- Starting BoolQ Fine-Tuning on {device.upper()} ---")
    best_acc = 0.0
    best_save_path = os.path.join(save_dir, "boolq_best_model.pt")

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch in pbar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            logits, loss = model(input_ids, attention_mask, labels=labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_train_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = total_train_loss / len(train_loader)
        val_acc, val_loss = evaluate_split(model, val_loader, device)

        print(f"Epoch {epoch+1} Summary:")
        print(f"  Train Loss: {avg_loss:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")
        print(f"  Val Acc:    {val_acc:.2f}%")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), best_save_path)
            print(f"  --> Saved new best BoolQ model ({val_acc:.2f}%) to {best_save_path}\n")
        else:
            print("\n")

    return best_save_path

# -----------------------------------------------------------------------
# 3. Final Combined Validation Testing Loop (GLUE & SuperGLUE)
# -----------------------------------------------------------------------

def prepare_sst2_val_loader(tokenizer, batch_size=64, max_length=128):
    dataset = load_dataset("glue", "sst2", split="validation")
    def tokenize_fn(examples):
        return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=max_length)
    tokenized = dataset.map(tokenize_fn, batched=True)
    tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    return DataLoader(tokenized, batch_size=batch_size)


def run_final_benchmark_suite(cfg, tokenizer, device, sst2_ckpt_path, boolq_ckpt_path):
    print("\n" + "=" * 60)
    print("      FINAL BENCHMARK EVALUATION (GLUE & SuperGLUE)      ")
    print("=" * 60)

    results = {}

    # 1. SST-2 Evaluation
    print("\n[1/2] Evaluating GLUE (SST-2)...")
    sst2_model = MambaForSequenceClassification(cfg, num_labels=2)
    if os.path.exists(sst2_ckpt_path):
        state_dict = torch.load(sst2_ckpt_path, map_location="cpu")
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        sst2_model.load_state_dict(state_dict)
        sst2_model.to(device)

        sst2_val_loader = prepare_sst2_val_loader(tokenizer)
        sst2_acc, sst2_loss = evaluate_split(sst2_model, sst2_val_loader, device)
        results["GLUE SST-2"] = sst2_acc
        print(f"--> SST-2 Validation Accuracy: {sst2_acc:.2f}% (Loss: {sst2_loss:.4f})")
    else:
        print(f"[Warning] SST-2 checkpoint not found at {sst2_ckpt_path}")

    # 2. BoolQ Evaluation
    print("\n[2/2] Evaluating SuperGLUE (BoolQ)...")
    boolq_model = MambaForSequenceClassification(cfg, num_labels=2)
    if os.path.exists(boolq_ckpt_path):
        state_dict = torch.load(boolq_ckpt_path, map_location="cpu")
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        boolq_model.load_state_dict(state_dict)
        boolq_model.to(device)

        _, boolq_val_loader = prepare_boolq_dataloaders(tokenizer, max_length=256)
        boolq_acc, boolq_loss = evaluate_split(boolq_model, boolq_val_loader, device)
        results["SuperGLUE BoolQ"] = boolq_acc
        print(f"--> BoolQ Validation Accuracy: {boolq_acc:.2f}% (Loss: {boolq_loss:.4f})")
    else:
        print(f"[Warning] BoolQ checkpoint not found at {boolq_ckpt_path}")

    # Summary Table
    print("\n" + "=" * 45)
    print(f"{'Benchmark Task':<25} | {'Validation Accuracy':<15}")
    print("-" * 45)
    for task, acc in results.items():
        print(f"{task:<25} | {acc:.2f}%")
    print("=" * 45 + "\n")

# -----------------------------------------------------------------------
# 4. Main Entry Point
# -----------------------------------------------------------------------

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    cfg = MambaConfig()

    # Step A: Load Pretrained WikiText Base Model for BoolQ fine-tuning
    base_ckpt_path = "/kaggle/input/models/ojasgupta074927/wikitext103-mamba-1-10500-32batch-size/pytorch/default/1/latest.pt"
    model = MambaForSequenceClassification(cfg, num_labels=2)

    if os.path.exists(base_ckpt_path):
        ckpt = torch.load(base_ckpt_path, map_location="cpu")
        state_dict = ckpt["model_state"]
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        base_state_dict = {k: v for k, v in state_dict.items() if "lm_head" not in k}
        model.load_state_dict(base_state_dict, strict=False)
        print("Loaded base pre-trained weights for BoolQ fine-tuning.")
    else:
        print("[Warning] Base checkpoint not found. Training from scratch.")

    # Step B: Fine-tune on BoolQ
    train_loader, val_loader = prepare_boolq_dataloaders(tokenizer, batch_size=32, max_length=256)
    boolq_best_path = train_boolq(model, train_loader, val_loader, device, epochs=3, lr=3e-5)

    # Step C: Run complete validation suite across both checkpoints
    sst2_best_path = "/kaggle/working/finetuned_checkpoints/best_model.pt"
    if not os.path.exists(sst2_best_path):
        sst2_best_path = "/kaggle/working/finetuned_models/best_model.pt"

    run_final_benchmark_suite(cfg, tokenizer, device, sst2_best_path, boolq_best_path)


if __name__ == "__main__":
    main()

Loaded base pre-trained weights for BoolQ fine-tuning.
Loading and tokenizing BoolQ dataset (SuperGLUE)...


Map:   0%|          | 0/9427 [00:00<?, ? examples/s]

Map:   0%|          | 0/3270 [00:00<?, ? examples/s]

Map:   0%|          | 0/3245 [00:00<?, ? examples/s]

Map:   0%|          | 0/9427 [00:00<?, ? examples/s]

Map:   0%|          | 0/3270 [00:00<?, ? examples/s]

Map:   0%|          | 0/3245 [00:00<?, ? examples/s]


--- Starting BoolQ Fine-Tuning on CUDA ---


Epoch 1/3:   0%|          | 0/295 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 24.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 18.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.41 GiB is allocated by PyTorch, and 11.09 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [9]:
import os
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

try:
    from mamba_ssm import Mamba as _CudaMamba
    HAS_CUDA_MAMBA = True
except ImportError:
    HAS_CUDA_MAMBA = False

# -----------------------------------------------------------------------
# 1. Unified Architecture Definition
# -----------------------------------------------------------------------

class TorchMamba(nn.Module):
    """Pure PyTorch fallback with identical parameter layout to mamba_ssm.Mamba."""
    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = int(self.expand * self.d_model)
        self.dt_rank = math.ceil(self.d_model / 16)

        self.in_proj = nn.Linear(self.d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            bias=True,
            kernel_size=d_conv,
            groups=self.d_inner,
            padding=d_conv - 1,
        )
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + self.d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, D = x.shape
        xz = self.in_proj(x)
        x_in, z = xz.chunk(2, dim=-1)

        x_in = x_in.transpose(1, 2)
        x_in = self.conv1d(x_in)[:, :, :L]
        x_in = x_in.transpose(1, 2)
        x_in = F.silu(x_in)

        x_dbl = self.x_proj(x_in)
        dt, B_mat, C_mat = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt))

        A = -torch.exp(self.A_log.float())
        D_mat = self.D.float()

        h = torch.zeros(B, self.d_inner, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dt_t = dt[:, t].unsqueeze(-1)
            dA = torch.exp(dt_t * A)
            dBx = dt_t * B_mat[:, t].unsqueeze(1) * x_in[:, t].unsqueeze(-1)
            h = dA * h + dBx
            y_t = (h * C_mat[:, t].unsqueeze(1)).sum(dim=-1) + D_mat * x_in[:, t]
            ys.append(y_t)

        y = torch.stack(ys, dim=1)
        out = y * F.silu(z)
        return self.out_proj(out)


class MambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        if HAS_CUDA_MAMBA:
            self.mamba = _CudaMamba(d_model=d_model, d_state=d_state, expand=expand)
        else:
            self.mamba = TorchMamba(d_model=d_model, d_state=d_state, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        if HAS_CUDA_MAMBA and not x.is_cuda:
            out = self._fallback_cpu_forward(h)
        else:
            out = self.mamba(h)
        return x + out

    def _fallback_cpu_forward(self, x: torch.Tensor) -> torch.Tensor:
        m = self.mamba
        B, L, D = x.shape
        xz = m.in_proj(x)
        x_in, z = xz.chunk(2, dim=-1)
        x_in = x_in.transpose(1, 2)
        x_in = F.pad(x_in, (m.d_conv - 1, 0))
        x_in = m.conv1d(x_in)[:, :, :L]
        x_in = x_in.transpose(1, 2)
        x_in = F.silu(x_in)

        x_proj = m.x_proj(x_in)
        dt, B_mat, C_mat = torch.split(x_proj, [m.dt_rank, m.d_state, m.d_state], dim=-1)
        dt = F.softplus(m.dt_proj(dt))

        A = -torch.exp(m.A_log.float())
        D_mat = m.D.float()
        h = torch.zeros(B, m.d_inner, m.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dt_t = dt[:, t].unsqueeze(-1)
            dA = torch.exp(dt_t * A)
            dBx = dt_t * B_mat[:, t].unsqueeze(1) * x_in[:, t].unsqueeze(-1)
            h = dA * h + dBx
            y_t = (h * C_mat[:, t].unsqueeze(1)).sum(dim=-1) + D_mat * x_in[:, t]
            ys.append(y_t)
        y = torch.stack(ys, dim=1)
        return m.out_proj(y * F.silu(z))


@dataclass
class MambaConfig:
    vocab_size: int = 50257
    d_model: int = 384
    n_layer: int = 16
    d_state: int = 16
    expand: int = 2


class MambaForSequenceClassification(nn.Module):
    def __init__(self, cfg: MambaConfig, num_labels: int = 2):
        super().__init__()
        self.cfg = cfg
        self.num_labels = num_labels

        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([
            MambaBlock(cfg.d_model, cfg.d_state, cfg.expand)
            for _ in range(cfg.n_layer)
        ])
        self.norm_out = nn.LayerNorm(cfg.d_model)
        self.classifier = nn.Linear(cfg.d_model, num_labels, bias=False)

        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, labels: Optional[torch.Tensor] = None):
        x = self.embed(input_ids)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm_out(x)

        sequence_lengths = (attention_mask.sum(dim=1) - 1).clamp(min=0)
        pooled_logits = x[torch.arange(x.shape[0], device=x.device), sequence_lengths]
        logits = self.classifier(pooled_logits)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return logits, loss

# -----------------------------------------------------------------------
# 2. BoolQ Dataset Pipeline & Fine-Tuning
# -----------------------------------------------------------------------

def prepare_boolq_dataloaders(tokenizer, batch_size=8, max_length=256):
    """Reduced batch size to 8 to prevent VRAM OOM errors during longer sequence training."""
    print("Loading and tokenizing BoolQ dataset (SuperGLUE)...")
    dataset = load_dataset("super_glue", "boolq")

    def tokenize_fn(examples):
        texts = [f"Question: {q} Passage: {p}" for q, p in zip(examples["question"], examples["passage"])]
        return tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=max_length
        )

    tokenized = dataset.map(tokenize_fn, batched=True)
    tokenized = tokenized.map(lambda x: {"label": int(x["label"])})
    tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

    train_loader = DataLoader(tokenized["train"], shuffle=True, batch_size=batch_size)
    val_loader = DataLoader(tokenized["validation"], batch_size=batch_size)
    return train_loader, val_loader


@torch.no_grad()
def evaluate_split(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits, loss = model(input_ids, attention_mask, labels=labels)
        if loss is not None:
            total_loss += loss.item()

        preds = torch.argmax(logits, dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    accuracy = (correct / total) * 100
    avg_loss = total_loss / max(1, len(dataloader))
    return accuracy, avg_loss


def train_boolq(model, train_loader, val_loader, device, epochs=3, lr=3e-5, accum_steps=4, save_dir="/kaggle/working/finetuned_checkpoints"):
    os.makedirs(save_dir, exist_ok=True)
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    # Adjust total steps for scheduler based on accumulation
    effective_steps = (len(train_loader) // accum_steps) * epochs
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, total_steps=effective_steps, pct_start=0.1
    )

    print(f"\n--- Starting BoolQ Fine-Tuning on {device.upper()} ---")
    best_acc = 0.0
    best_save_path = os.path.join(save_dir, "boolq_best_model.pt")

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        optimizer.zero_grad()

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for i, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits, loss = model(input_ids, attention_mask, labels=labels)
            
            # Scale loss for gradient accumulation
            loss = loss / accum_steps
            loss.backward()
            
            total_train_loss += loss.item() * accum_steps # Track actual loss

            if (i + 1) % accum_steps == 0 or (i + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            pbar.set_postfix({"loss": f"{(loss.item() * accum_steps):.4f}"})

        avg_loss = total_train_loss / len(train_loader)
        val_acc, val_loss = evaluate_split(model, val_loader, device)

        print(f"Epoch {epoch+1} Summary:")
        print(f"  Train Loss: {avg_loss:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")
        print(f"  Val Acc:    {val_acc:.2f}%")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), best_save_path)
            print(f"  --> Saved new best BoolQ model ({val_acc:.2f}%) to {best_save_path}\n")
        else:
            print("\n")

    return best_save_path

# -----------------------------------------------------------------------
# 3. Final Combined Validation Testing Loop (GLUE & SuperGLUE)
# -----------------------------------------------------------------------

def prepare_sst2_val_loader(tokenizer, batch_size=32, max_length=128):
    dataset = load_dataset("glue", "sst2", split="validation")
    def tokenize_fn(examples):
        return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=max_length)
    tokenized = dataset.map(tokenize_fn, batched=True)
    tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    return DataLoader(tokenized, batch_size=batch_size)


def run_final_benchmark_suite(cfg, tokenizer, device, sst2_ckpt_path, boolq_ckpt_path):
    print("\n" + "=" * 60)
    print("      FINAL BENCHMARK EVALUATION (GLUE & SuperGLUE)      ")
    print("=" * 60)

    results = {}

    # 1. SST-2 Evaluation
    print("\n[1/2] Evaluating GLUE (SST-2)...")
    sst2_model = MambaForSequenceClassification(cfg, num_labels=2)
    if os.path.exists(sst2_ckpt_path):
        state_dict = torch.load(sst2_ckpt_path, map_location="cpu")
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        sst2_model.load_state_dict(state_dict)
        sst2_model.to(device)

        sst2_val_loader = prepare_sst2_val_loader(tokenizer)
        sst2_acc, sst2_loss = evaluate_split(sst2_model, sst2_val_loader, device)
        results["GLUE SST-2"] = sst2_acc
        print(f"--> SST-2 Validation Accuracy: {sst2_acc:.2f}% (Loss: {sst2_loss:.4f})")
    else:
        print(f"[Warning] SST-2 checkpoint not found at {sst2_ckpt_path}")

    # 2. BoolQ Evaluation
    print("\n[2/2] Evaluating SuperGLUE (BoolQ)...")
    boolq_model = MambaForSequenceClassification(cfg, num_labels=2)
    if os.path.exists(boolq_ckpt_path):
        state_dict = torch.load(boolq_ckpt_path, map_location="cpu")
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        boolq_model.load_state_dict(state_dict)
        boolq_model.to(device)

        _, boolq_val_loader = prepare_boolq_dataloaders(tokenizer, batch_size=32, max_length=256)
        boolq_acc, boolq_loss = evaluate_split(boolq_model, boolq_val_loader, device)
        results["SuperGLUE BoolQ"] = boolq_acc
        print(f"--> BoolQ Validation Accuracy: {boolq_acc:.2f}% (Loss: {boolq_loss:.4f})")
    else:
        print(f"[Warning] BoolQ checkpoint not found at {boolq_ckpt_path}")

    # Summary Table
    print("\n" + "=" * 45)
    print(f"{'Benchmark Task':<25} | {'Validation Accuracy':<15}")
    print("-" * 45)
    for task, acc in results.items():
        print(f"{task:<25} | {acc:.2f}%")
    print("=" * 45 + "\n")

# -----------------------------------------------------------------------
# 4. Main Entry Point
# -----------------------------------------------------------------------

import gc

def main():
    # 1. Aggressively destroy old references
    if 'model' in globals():
        del globals()['model']
    if 'sst2_model' in globals():
        del globals()['sst2_model']
        
    # 2. Force Python to clean up destroyed objects
    gc.collect()
    
    # 3. Now clear the GPU cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    cfg = MambaConfig()

    # Step A: Load Pretrained WikiText Base Model for BoolQ fine-tuning
    base_ckpt_path = "/kaggle/input/models/ojasgupta074927/wikitext103-mamba-1-10500-32batch-size/pytorch/default/1/latest.pt"
    model = MambaForSequenceClassification(cfg, num_labels=2)

    if os.path.exists(base_ckpt_path):
        ckpt = torch.load(base_ckpt_path, map_location="cpu")
        state_dict = ckpt["model_state"]
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        base_state_dict = {k: v for k, v in state_dict.items() if "lm_head" not in k}
        model.load_state_dict(base_state_dict, strict=False)
        print("Loaded base pre-trained weights for BoolQ fine-tuning.")
    else:
        print("[Warning] Base checkpoint not found. Training from scratch.")

    # Step B: Fine-tune on BoolQ (Batch size 8 with 4 accum steps = effective batch of 32)
    train_loader, val_loader = prepare_boolq_dataloaders(tokenizer, batch_size=8, max_length=256)
    boolq_best_path = train_boolq(model, train_loader, val_loader, device, epochs=3, lr=3e-5, accum_steps=4)

    # Step C: Run complete validation suite across both checkpoints
    sst2_best_path = "/kaggle/working/finetuned_checkpoints/best_model.pt"
    if not os.path.exists(sst2_best_path):
        sst2_best_path = "/kaggle/working/finetuned_models/best_model.pt"

    run_final_benchmark_suite(cfg, tokenizer, device, sst2_best_path, boolq_best_path)


if __name__ == "__main__":
    main()

Loaded base pre-trained weights for BoolQ fine-tuning.
Loading and tokenizing BoolQ dataset (SuperGLUE)...

--- Starting BoolQ Fine-Tuning on CUDA ---


Epoch 1/3: 100%|██████████| 1179/1179 [02:10<00:00,  9.05it/s, loss=0.6141]


Epoch 1 Summary:
  Train Loss: 0.6903
  Val Loss:   0.6501
  Val Acc:    63.61%
  --> Saved new best BoolQ model (63.61%) to /kaggle/working/finetuned_checkpoints/boolq_best_model.pt



Epoch 2/3: 100%|██████████| 1179/1179 [02:07<00:00,  9.24it/s, loss=0.4347]


Epoch 2 Summary:
  Train Loss: 0.5441
  Val Loss:   0.6537
  Val Acc:    64.10%
  --> Saved new best BoolQ model (64.10%) to /kaggle/working/finetuned_checkpoints/boolq_best_model.pt



Epoch 3/3:  99%|█████████▉| 1171/1179 [02:07<00:00,  9.16it/s, loss=0.1286]


ValueError: Tried to step 883 times. The specified number of total steps is 882

In [8]:
try:
    from mamba_ssm import Mamba
    import causal_conv1d
    print("✅ Fast CUDA Kernels successfully loaded!")
except Exception as e:
    print(f"❌ Failed to load fast kernels: {e}")

✅ Fast CUDA Kernels successfully loaded!


In [7]:
!pip install causal-conv1d --no-build-isolation
!pip install mamba-ssm --no-build-isolation

  Preparing metadata (pyproject.toml) ... done
  Created wheel for causal-conv1d: filename=causal_conv1d-1.7.0-cp312-cp312-linux_x86_64.whl size=196558813 sha256=8e81f8c76435ad31aa41edc6c0c9d26de971a293b190183d2816da71547614d7
  Stored in directory: /root/.cache/pip/wheels/6f/0b/1b/e063b1e314a5ff0783e450525921cb8f5c48fbb03639bad6ba
Successfully built causal-conv1d
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 11.2 MB/s eta 0:00:00
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 47.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 MB 21.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.4/772.4 kB 35.5 MB/s eta 0:00:00
   ━━━━━━

In [10]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

try:
    from mamba_ssm import Mamba as _CudaMamba
    HAS_CUDA_MAMBA = True
except ImportError:
    HAS_CUDA_MAMBA = False

# -----------------------------------------------------------------------
# 1. Architecture Definitions (Required for Loading Weights)
# -----------------------------------------------------------------------

class TorchMamba(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand = expand
        self.d_inner = int(self.expand * self.d_model)
        self.dt_rank = math.ceil(self.d_model / 16)

        self.in_proj = nn.Linear(self.d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner, out_channels=self.d_inner, bias=True,
            kernel_size=d_conv, groups=self.d_inner, padding=d_conv - 1,
        )
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + self.d_state * 2, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        A = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, D = x.shape
        xz = self.in_proj(x)
        x_in, z = xz.chunk(2, dim=-1)

        x_in = x_in.transpose(1, 2)
        x_in = self.conv1d(x_in)[:, :, :L]
        x_in = x_in.transpose(1, 2)
        x_in = F.silu(x_in)

        x_dbl = self.x_proj(x_in)
        dt, B_mat, C_mat = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt))

        A = -torch.exp(self.A_log.float())
        D_mat = self.D.float()

        h = torch.zeros(B, self.d_inner, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dt_t = dt[:, t].unsqueeze(-1)
            dA = torch.exp(dt_t * A)
            dBx = dt_t * B_mat[:, t].unsqueeze(1) * x_in[:, t].unsqueeze(-1)
            h = dA * h + dBx
            y_t = (h * C_mat[:, t].unsqueeze(1)).sum(dim=-1) + D_mat * x_in[:, t]
            ys.append(y_t)

        y = torch.stack(ys, dim=1)
        out = y * F.silu(z)
        return self.out_proj(out)


class MambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        if HAS_CUDA_MAMBA:
            self.mamba = _CudaMamba(d_model=d_model, d_state=d_state, expand=expand)
        else:
            self.mamba = TorchMamba(d_model=d_model, d_state=d_state, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        if HAS_CUDA_MAMBA and not x.is_cuda:
            out = self._fallback_cpu_forward(h)
        else:
            out = self.mamba(h)
        return x + out

    def _fallback_cpu_forward(self, x: torch.Tensor) -> torch.Tensor:
        m = self.mamba
        B, L, D = x.shape
        xz = m.in_proj(x)
        x_in, z = xz.chunk(2, dim=-1)
        x_in = x_in.transpose(1, 2)
        x_in = F.pad(x_in, (m.d_conv - 1, 0))
        x_in = m.conv1d(x_in)[:, :, :L]
        x_in = x_in.transpose(1, 2)
        x_in = F.silu(x_in)

        x_proj = m.x_proj(x_in)
        dt, B_mat, C_mat = torch.split(x_proj, [m.dt_rank, m.d_state, m.d_state], dim=-1)
        dt = F.softplus(m.dt_proj(dt))

        A = -torch.exp(m.A_log.float())
        D_mat = m.D.float()
        h = torch.zeros(B, m.d_inner, m.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(L):
            dt_t = dt[:, t].unsqueeze(-1)
            dA = torch.exp(dt_t * A)
            dBx = dt_t * B_mat[:, t].unsqueeze(1) * x_in[:, t].unsqueeze(-1)
            h = dA * h + dBx
            y_t = (h * C_mat[:, t].unsqueeze(1)).sum(dim=-1) + D_mat * x_in[:, t]
            ys.append(y_t)
        y = torch.stack(ys, dim=1)
        return m.out_proj(y * F.silu(z))


@dataclass
class MambaConfig:
    vocab_size: int = 50257
    d_model: int = 384
    n_layer: int = 16
    d_state: int = 16
    expand: int = 2


class MambaForSequenceClassification(nn.Module):
    def __init__(self, cfg: MambaConfig, num_labels: int = 2):
        super().__init__()
        self.cfg = cfg
        self.num_labels = num_labels

        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([
            MambaBlock(cfg.d_model, cfg.d_state, cfg.expand)
            for _ in range(cfg.n_layer)
        ])
        self.norm_out = nn.LayerNorm(cfg.d_model)
        self.classifier = nn.Linear(cfg.d_model, num_labels, bias=False)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, labels: Optional[torch.Tensor] = None):
        x = self.embed(input_ids)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm_out(x)

        sequence_lengths = (attention_mask.sum(dim=1) - 1).clamp(min=0)
        pooled_logits = x[torch.arange(x.shape[0], device=x.device), sequence_lengths]
        logits = self.classifier(pooled_logits)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)

        return logits, loss

# -----------------------------------------------------------------------
# 2. Dataloaders
# -----------------------------------------------------------------------

def prepare_sst2_val_loader(tokenizer, batch_size=64, max_length=128):
    dataset = load_dataset("glue", "sst2", split="validation")
    def tokenize_fn(examples):
        return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=max_length)
    tokenized = dataset.map(tokenize_fn, batched=True)
    tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    return DataLoader(tokenized, batch_size=batch_size)

def prepare_boolq_val_loader(tokenizer, batch_size=64, max_length=256):
    dataset = load_dataset("super_glue", "boolq", split="validation")
    def tokenize_fn(examples):
        texts = [f"Question: {q} Passage: {p}" for q, p in zip(examples["question"], examples["passage"])]
        return tokenizer(texts, padding="max_length", truncation=True, max_length=max_length)
    tokenized = dataset.map(tokenize_fn, batched=True)
    tokenized = tokenized.map(lambda x: {"label": int(x["label"])})
    tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    return DataLoader(tokenized, batch_size=batch_size)

# -----------------------------------------------------------------------
# 3. Evaluation Logic
# -----------------------------------------------------------------------

@torch.no_grad()
def evaluate_split(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0

    for batch in tqdm(dataloader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits, _ = model(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=-1)
        
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return (correct / total) * 100

def load_model(ckpt_path, cfg, device):
    model = MambaForSequenceClassification(cfg, num_labels=2)
    if os.path.exists(ckpt_path):
        state_dict = torch.load(ckpt_path, map_location="cpu")
        state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(state_dict, strict=True)
        model.to(device)
        print(f"Loaded weights from {ckpt_path}")
        return model
    else:
        print(f"[Error] Checkpoint not found at {ckpt_path}")
        return None

def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    cfg = MambaConfig()
    results = {}

    print("\n" + "=" * 60)
    print("      FINAL BENCHMARK EVALUATION (GLUE & SuperGLUE)      ")
    print("=" * 60)

    # 1. SST-2 Evaluation
    print("\n[1/2] Evaluating GLUE (SST-2)...")
    sst2_ckpt = "/kaggle/working/finetuned_checkpoints/best_model.pt"
    if not os.path.exists(sst2_ckpt):
        sst2_ckpt = "/kaggle/working/latest_model.pt" # Fallback to your copied path
        
    sst2_model = load_model(sst2_ckpt, cfg, device)
    if sst2_model:
        sst2_loader = prepare_sst2_val_loader(tokenizer)
        acc = evaluate_split(sst2_model, sst2_loader, device)
        results["GLUE SST-2"] = acc
        print(f"--> SST-2 Validation Accuracy: {acc:.2f}%")

    # 2. BoolQ Evaluation
    print("\n[2/2] Evaluating SuperGLUE (BoolQ)...")
    boolq_ckpt = "/kaggle/working/finetuned_checkpoints/boolq_best_model.pt"
    boolq_model = load_model(boolq_ckpt, cfg, device)
    if boolq_model:
        boolq_loader = prepare_boolq_val_loader(tokenizer)
        acc = evaluate_split(boolq_model, boolq_loader, device)
        results["SuperGLUE BoolQ"] = acc
        print(f"--> BoolQ Validation Accuracy: {acc:.2f}%")

    # Summary Table
    print("\n" + "=" * 45)
    print(f"{'Benchmark Task':<25} | {'Validation Accuracy':<15}")
    print("-" * 45)
    for task, acc in results.items():
        print(f"{task:<25} | {acc:.2f}%")
    print("=" * 45 + "\n")

if __name__ == "__main__":
    main()


      FINAL BENCHMARK EVALUATION (GLUE & SuperGLUE)      

[1/2] Evaluating GLUE (SST-2)...
Loaded weights from /kaggle/working/finetuned_checkpoints/best_model.pt


Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Evaluating: 100%|██████████| 14/14 [00:02<00:00,  6.83it/s]


--> SST-2 Validation Accuracy: 86.81%

[2/2] Evaluating SuperGLUE (BoolQ)...
Loaded weights from /kaggle/working/finetuned_checkpoints/boolq_best_model.pt


Map:   0%|          | 0/3270 [00:00<?, ? examples/s]

Map:   0%|          | 0/3270 [00:00<?, ? examples/s]

Evaluating: 100%|██████████| 52/52 [00:16<00:00,  3.23it/s]

--> BoolQ Validation Accuracy: 64.10%

Benchmark Task            | Validation Accuracy
---------------------------------------------
GLUE SST-2                | 86.81%
SuperGLUE BoolQ           | 64.10%



In [11]:
import os
import shutil

# Model paths
model1 = "/kaggle/working/finetuned_checkpoints/best_model.pt"
model2 = "/kaggle/working/finetuned_checkpoints/boolq_best_model.pt"

# Temporary folder containing both models
temp_dir = "/kaggle/working/fine_tuned_models"
os.makedirs(temp_dir, exist_ok=True)

# Copy both models into the folder
shutil.copy2(model1, os.path.join(temp_dir, "best_model.pt"))
shutil.copy2(model2, os.path.join(temp_dir, "boolq_best_model.pt"))

# Create ZIP
zip_base = "/kaggle/working/fine_tuned_models"
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir=temp_dir
)

print("Created:", zip_path)
print(f"Size: {os.path.getsize(zip_path) / (1024**2):.2f} MB")

# Ensure ZIP is in /kaggle/working/
cwd_zip = "/kaggle/working/fine_tuned_models.zip"

if zip_path != cwd_zip:
    shutil.copy2(zip_path, cwd_zip)

print("Final ZIP:", cwd_zip)
print("\nContents:")
import zipfile
with zipfile.ZipFile(cwd_zip, "r") as z:
    for name in z.namelist():
        print(" -", name)

Created: /kaggle/working/fine_tuned_models.zip
Size: 246.98 MB
Final ZIP: /kaggle/working/fine_tuned_models.zip

Contents:
 - best_model.pt
 - boolq_best_model.pt
